In [1]:
import os
import re
from urllib.parse import urljoin, urlparse
import requests
from bs4 import BeautifulSoup
import zipfile
from io import TextIOWrapper

import pandas as pd
from Bio import SeqIO

In [2]:
#variables

class_name = "aves".lower()
genbank_name = "GenBank269_2025-12-09".lower()
save_dir = "../data"

In [4]:
midori_url = "https://www.reference-midori.info/"
download_page = urljoin(midori_url, "download.php")

In [53]:
os.makedirs(save_dir, exist_ok=True)

In [54]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

In [55]:
# Загружаем страницу
resp = requests.get(download_page, headers=headers, timeout=60)
resp.raise_for_status()

# Парсим ссылки
soup = BeautifulSoup(resp.text, "html.parser")

links = []
for a in soup.find_all("a", href=True):
    href = a["href"]
    full_url = urljoin(download_page, href)
    text = a.get_text(" ", strip=True)
    links.append((text, full_url))

# Оставляем только .fasta.zip
fasta_zip_links = []
for text, url in links:
    if url.lower().endswith(".fasta.zip"):
        fasta_zip_links.append((text, url))

print(f"Всего .fasta.zip ссылок: {len(fasta_zip_links)}")

Всего .fasta.zip ссылок: 4402


In [56]:
# Разделяем на группы и отбираем по genbank_name
longest_links = []
uniq_links = []

for text, url in fasta_zip_links:
    path = urlparse(url).path.lower()
    if genbank_name+'/blast/' in path:
        if "/longest/fasta/" in path:
            longest_links.append((text, url))
        elif "/uniq/fasta/" in path:
            uniq_links.append((text, url))

In [57]:
print(f"longest/fasta: {len(longest_links)}")
print(f"uniq/fasta: {len(uniq_links)}")

longest/fasta: 15
uniq/fasta: 15


In [59]:
def download_links(link_list, subfolder):
    folder = os.path.join(save_dir, subfolder)
    os.makedirs(folder, exist_ok=True)

    for i, (text, url) in enumerate(link_list, start=1):
        filename = os.path.basename(urlparse(url).path)
        out_path = os.path.join(folder, filename)

        print(f"[{i}/{len(link_list)}] Скачиваю {filename}")
        with requests.get(url, headers=headers, stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(out_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)

# 8. Сначала longest/fasta, потом uniq/fasta
download_links(longest_links, genbank_name+"_longest_fasta")

[1/15] Скачиваю MIDORI2_LONGEST_NUC_GB269_A6_BLAST.fasta.zip
[2/15] Скачиваю MIDORI2_LONGEST_NUC_GB269_A8_BLAST.fasta.zip
[3/15] Скачиваю MIDORI2_LONGEST_NUC_GB269_CO1_BLAST.fasta.zip
[4/15] Скачиваю MIDORI2_LONGEST_NUC_GB269_CO2_BLAST.fasta.zip
[5/15] Скачиваю MIDORI2_LONGEST_NUC_GB269_CO3_BLAST.fasta.zip
[6/15] Скачиваю MIDORI2_LONGEST_NUC_GB269_Cytb_BLAST.fasta.zip
[7/15] Скачиваю MIDORI2_LONGEST_NUC_GB269_ND1_BLAST.fasta.zip
[8/15] Скачиваю MIDORI2_LONGEST_NUC_GB269_ND2_BLAST.fasta.zip
[9/15] Скачиваю MIDORI2_LONGEST_NUC_GB269_ND3_BLAST.fasta.zip
[10/15] Скачиваю MIDORI2_LONGEST_NUC_GB269_ND4L_BLAST.fasta.zip
[11/15] Скачиваю MIDORI2_LONGEST_NUC_GB269_ND4_BLAST.fasta.zip
[12/15] Скачиваю MIDORI2_LONGEST_NUC_GB269_ND5_BLAST.fasta.zip
[13/15] Скачиваю MIDORI2_LONGEST_NUC_GB269_ND6_BLAST.fasta.zip
[14/15] Скачиваю MIDORI2_LONGEST_NUC_GB269_lrRNA_BLAST.fasta.zip
[15/15] Скачиваю MIDORI2_LONGEST_NUC_GB269_srRNA_BLAST.fasta.zip


In [120]:
download_links(uniq_links, genbank_name+"_uniq_fasta")

[1/15] Скачиваю MIDORI2_UNIQ_NUC_GB269_A6_BLAST.fasta.zip
[2/15] Скачиваю MIDORI2_UNIQ_NUC_GB269_A8_BLAST.fasta.zip
[3/15] Скачиваю MIDORI2_UNIQ_NUC_GB269_CO1_BLAST.fasta.zip
[4/15] Скачиваю MIDORI2_UNIQ_NUC_GB269_CO2_BLAST.fasta.zip
[5/15] Скачиваю MIDORI2_UNIQ_NUC_GB269_CO3_BLAST.fasta.zip
[6/15] Скачиваю MIDORI2_UNIQ_NUC_GB269_Cytb_BLAST.fasta.zip
[7/15] Скачиваю MIDORI2_UNIQ_NUC_GB269_ND1_BLAST.fasta.zip
[8/15] Скачиваю MIDORI2_UNIQ_NUC_GB269_ND2_BLAST.fasta.zip
[9/15] Скачиваю MIDORI2_UNIQ_NUC_GB269_ND3_BLAST.fasta.zip
[10/15] Скачиваю MIDORI2_UNIQ_NUC_GB269_ND4L_BLAST.fasta.zip
[11/15] Скачиваю MIDORI2_UNIQ_NUC_GB269_ND4_BLAST.fasta.zip
[12/15] Скачиваю MIDORI2_UNIQ_NUC_GB269_ND5_BLAST.fasta.zip
[13/15] Скачиваю MIDORI2_UNIQ_NUC_GB269_ND6_BLAST.fasta.zip
[14/15] Скачиваю MIDORI2_UNIQ_NUC_GB269_lrRNA_BLAST.fasta.zip
[15/15] Скачиваю MIDORI2_UNIQ_NUC_GB269_srRNA_BLAST.fasta.zip


In [4]:
# Папки с уже скачанными архивами
longest_dir = "../data/"+genbank_name+"_longest_fasta"
uniq_dir = "../data/"+genbank_name+"_uniq_fasta"

In [5]:
def read_fasta_from_zip(zip_path: str):
    with zipfile.ZipFile(zip_path, "r") as zf:
        fasta_names = [
            name for name in zf.namelist()
            if name.lower().endswith((".fasta", ".fa", ".fas"))
        ]

        for fasta_name in fasta_names:
            with zf.open(fasta_name) as fh:
                text_fh = TextIOWrapper(fh, encoding="utf-8", errors="replace")
                for record in SeqIO.parse(text_fh, "fasta"):
                    yield record

In [10]:
def process_folder(folder: str) -> pd.DataFrame:
    rows = []

    zip_files = sorted(
        [f for f in os.listdir(folder) if f.lower().endswith(".fasta.zip")]
    )

    for i, zip_name in enumerate(zip_files, start=1):
        zip_path = os.path.join(folder, zip_name)
        gene = zip_name.split("_")[-2]

        print(f"[{i}/{len(zip_files)}] {zip_name} -> gene={gene}")

        if class_name == "all":
            try:
                for record in read_fasta_from_zip(zip_path):
                    header = record.description
                    split_name = header.split(";")[-1].split("_")[0:2]
                    name = "_".join(split_name)
                    split_class_name = header.split(";")[3].split("_")[:-1]
                    selected_class_name = "_".join(split_class_name)
                    rows.append({
                        "class": selected_class_name,
                        "species_name": name,
                        "gen": gene,
                        "gen_seq": str(record.seq)
                    })

            except Exception as e:
                print(f"Ошибка в {zip_name}: {e}")
        else:
            try:
                for record in read_fasta_from_zip(zip_path):
                    header = record.description
                    split_class_name = header.split(";")[3].split("_")[:-1]
                    selected_class_name = "_".join(split_class_name).lower()
                    if selected_class_name == class_name:
                        split_name = header.split(";")[-1].split("_")[0:2]
                        name = "_".join(split_name)

                        rows.append({
                            "species_name": name,
                            "gen": gene,
                            "gen_seq": str(record.seq)
                        })

            except Exception as e:
                print(f"Ошибка в {zip_name}: {e}")

    return pd.DataFrame(rows)

In [11]:
df_longest = process_folder(longest_dir)
df_uniq = process_folder(uniq_dir)

MIDORI2_LONGEST_NUC_GB269_A6_BLAST.fasta.zip
[1/15] MIDORI2_LONGEST_NUC_GB269_A6_BLAST.fasta.zip -> gene=A6
MIDORI2_LONGEST_NUC_GB269_A8_BLAST.fasta.zip
[2/15] MIDORI2_LONGEST_NUC_GB269_A8_BLAST.fasta.zip -> gene=A8
MIDORI2_LONGEST_NUC_GB269_CO1_BLAST.fasta.zip
[3/15] MIDORI2_LONGEST_NUC_GB269_CO1_BLAST.fasta.zip -> gene=CO1
MIDORI2_LONGEST_NUC_GB269_CO2_BLAST.fasta.zip
[4/15] MIDORI2_LONGEST_NUC_GB269_CO2_BLAST.fasta.zip -> gene=CO2
MIDORI2_LONGEST_NUC_GB269_CO3_BLAST.fasta.zip
[5/15] MIDORI2_LONGEST_NUC_GB269_CO3_BLAST.fasta.zip -> gene=CO3
MIDORI2_LONGEST_NUC_GB269_Cytb_BLAST.fasta.zip
[6/15] MIDORI2_LONGEST_NUC_GB269_Cytb_BLAST.fasta.zip -> gene=Cytb
MIDORI2_LONGEST_NUC_GB269_ND1_BLAST.fasta.zip
[7/15] MIDORI2_LONGEST_NUC_GB269_ND1_BLAST.fasta.zip -> gene=ND1
MIDORI2_LONGEST_NUC_GB269_ND2_BLAST.fasta.zip
[8/15] MIDORI2_LONGEST_NUC_GB269_ND2_BLAST.fasta.zip -> gene=ND2
MIDORI2_LONGEST_NUC_GB269_ND3_BLAST.fasta.zip
[9/15] MIDORI2_LONGEST_NUC_GB269_ND3_BLAST.fasta.zip -> gene=ND3
MIDO

In [131]:
df_longest.to_csv(f"{save_dir}/{genbank_name}_{class_name}_longest.csv", index=False)
df_uniq.to_csv(f"{save_dir}/{genbank_name}_{class_name}_uniq.csv", index=False)

In [128]:
df_longest

,species_name,gen,gen_seq
0,Ciconia_boyciana,A6,ATGAACCTAAGCTTCTTCGATCAATTCGCCAGCCCATGCCTACTAG...
1,Ciconia_ciconia,A6,ATGAACCTAAGCTTCTTCGATCAATTCGCCAGCCCATGCCTACTAG...
2,Ciconia_maguari,A6,ATGAACCTAAGCTTCTTCGACCAATTCGCTAGCCCATATCTATTAG...
3,Ciconia_nigra,A6,ATGAATCTAAGCTTCTTCGACCAATTCACCAGCCCATGCCTACTAG...
4,Attagis_gayi,A6,ATGAACCTAAGCTTCTTCGACCAATTTACAAGCCCATGCCTTCTAG...
...,...,...,...
49399,Nothoprocta_pentlandii,srRNA,AAAAGACTTAGTCCTAACCTTACCATTAATTATTACTAAACATATA...
49400,Nothoprocta_perdicaria,srRNA,AAAAGACTTAGTCCTAACCTTACCATTAATTATTACTAAACATATA...
49401,Nothura_maculosa,srRNA,GCTTAGCCCTAAATCTAAATGCTTACCTTACTTAAGCATTCGCCCG...
49402,Tinamus_guttatus,srRNA,AAAAGACTTAGTCCTAACCTTACCATTAATTATCATTAGATATATA...
